In [1]:
import os
import random

import numpy as np
import pandas as pd

from PIL import Image

import torch
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T

# Dataset & DataLoader

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this script because the submission block uses .cuda().")

In [3]:
def _resolve_path(root_path: str, filename: str) -> str:
    """Find filename in root_path or common parent layout."""
    candidates = [
        os.path.join(root_path, filename),
        os.path.join(root_path, ".", filename),
        os.path.join(root_path, "..", filename),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return os.path.abspath(p)
    raise FileNotFoundError(f"Could not find {filename} under {root_path} (or its parent).")


def _resolve_dir(root_path: str, dirname: str) -> str:
    """Find image directory in root_path or accept root_path already being that directory."""
    candidates = [
        os.path.join(root_path, dirname),
        os.path.join(root_path, ".", dirname),
        os.path.abspath(root_path),
    ]
    for p in candidates:
        if os.path.isdir(p) and os.path.basename(os.path.normpath(p)).lower() == dirname.lower():
            return os.path.abspath(p)
    # common case: base root contains Train_64/Test_64
    p = os.path.join(root_path, dirname)
    if os.path.isdir(p):
        return os.path.abspath(p)
    raise FileNotFoundError(f"Could not find directory {dirname} under {root_path}.")


class TrainDataset(Dataset):
    """
    Expected dataset layout (recommended):
      {root_path}/Train_64.csv
      {root_path}/Train_64/<image files>
    """
    def __init__(self, root_path, transform, data_aug=None):
        super().__init__()
        self.root_path = root_path
        self.transform = transform
        self.augmentation = data_aug

        csv_path = _resolve_path(root_path, "Train_64.csv")
        self.annotation = pd.read_csv(csv_path)

        # robust extraction
        self.image_list = self.annotation.iloc[:, 0].to_numpy()
        self.labels = self.annotation.iloc[:, 1].to_numpy().astype(int)

        self.image_dir = _resolve_dir(root_path, "Train_64/Train_64")

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, str(self.image_list[index]))
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.augmentation is not None:
                img = self.augmentation(img)
            img = self.transform(img)
        return img, int(self.labels[index])


class TestDataset(Dataset):
    """
    Expected dataset layout (recommended):
      {root_path}/Test_64.csv
      {root_path}/Test_64/<image files>
    """
    def __init__(self, root_path, transform, data_aug=None):
        super().__init__()
        self.root_path = root_path
        self.transform = transform
        self.augmentation = data_aug

        csv_path = _resolve_path(root_path, "Test_64.csv")
        self.annotation = pd.read_csv(csv_path)

        self.image_list = self.annotation.iloc[:, 0].to_numpy()
        self.image_dir = _resolve_dir(root_path, "Test_64/Test_64")

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, index):
        img_path = os.path.join(self.image_dir, str(self.image_list[index]))
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            if self.augmentation is not None:
                img = self.augmentation(img)
            img = self.transform(img)
        return img

In [4]:
image_size = 64
batch_size_train = 64
batch_size_eval = 256

mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

val_ratio = 0.20
epochs = 30
label_smoothing = 0.1

save_path = "best_model.pth"

In [5]:
# data augmentation
train_transform = T.Compose([
    T.Pad(4, padding_mode='reflect'),
    T.RandomCrop(image_size),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(4),
    T.ColorJitter(brightness=0.02, contrast=0.02, saturation=0.01, hue=0.0),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean, std),
])

In [6]:
data_root = "./input/"

train_dataset_aug  = TrainDataset(root_path=data_root, transform=train_transform)
train_dataset_eval = TrainDataset(root_path=data_root, transform=eval_transform)
test_dataset = TestDataset(root_path=data_root, transform=eval_transform)

In [7]:
def stratified_split(labels: np.ndarray, val_ratio: float, seed: int):
    """Return (train_indices, val_indices) with per-class stratification.
    No sklearn dependency.
    """
    rng = np.random.default_rng(seed)
    labels = labels.astype(int)
    classes, counts = np.unique(labels, return_counts=True)

    val_indices = []
    train_indices = []

    for c, cnt in zip(classes, counts):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        n_val_c = int(round(cnt * val_ratio))
        # keep at least 1 sample in train if possible
        n_val_c = min(max(n_val_c, 1), max(cnt - 1, 1))
        val_indices.extend(idx[:n_val_c].tolist())
        train_indices.extend(idx[n_val_c:].tolist())

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices

In [8]:
labels = np.array(train_dataset_eval.labels, dtype=int)
train_indices, val_indices = stratified_split(labels, val_ratio=val_ratio, seed=SEED)

train_subset = Subset(train_dataset_aug, train_indices)
val_subset   = Subset(train_dataset_eval, val_indices)

num_workers = 0

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size_train,
    shuffle=True,
    num_workers=num_workers,
    drop_last=False,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size_eval,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=(num_workers > 0),
)

# Your Awesome Model

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
class SELayer(nn.Module):
    """Squeeze-and-Excitation Block"""
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class Res2NetSEBasicBlock(nn.Module):
    """
    Res2Net + SE Block이 통합된 BasicBlock
    - ResNeXt의 Grouped Conv 대신 일반 Conv 사용 (groups=1)
    """
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, scale=4, reduction=16):
        super(Res2NetSEBasicBlock, self).__init__()
        
        # ResNet BasicBlock의 첫 번째 3x3 Conv
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        
        self.scales = scale  # Res2Net의 스케일 차원 (채널 분할 개수)
        assert planes % self.scales == 0, "planes must be divisible by scale"
        self.planes_per_scale = planes // self.scales # 각 스케일 그룹의 채널 수

        # Res2Net의 3x3 Conv들 (일반 Conv)
        # s-1 개의 3x3 Conv가 필요하며, 입력/출력 채널은 planes_per_scale입니다.
        self.convs = nn.ModuleList()
        for i in range(self.scales - 1):
            self.convs.append(
                nn.Conv2d(self.planes_per_scale, self.planes_per_scale, kernel_size=3, stride=1, 
                          padding=1, groups=1, bias=False) # groups=1로 설정
            )
            
        self.bns = nn.ModuleList([nn.BatchNorm2d(self.planes_per_scale) for _ in range(self.scales - 1)])

        # 최종 3x3 Conv (BasicBlock의 마지막 Conv 역할)
        self.conv3 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes)
        
        # SE Block 추가
        self.se = SELayer(planes, reduction)

        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        # Res2Net Core
        # 1. 채널 분할 (Split)
        spx = torch.split(out, self.planes_per_scale, dim=1) # (B, C, H, W) -> (B, C/s, H, W) * s
        
        sp = []
        sp.append(spx[0]) # x1은 그대로 통과
        
        for i in range(self.scales - 1):
            # 2. 계층적 연결 (Hierarchical Connection)
            if i == 0:
                sp_i = spx[i+1] # x2
            else:
                sp_i = spx[i+1] + sp[i] # xi = xi + y_i-1
            
            sp_i = self.convs[i](sp_i)
            sp_i = self.bns[i](sp_i)
            sp_i = self.relu(sp_i)
            sp.append(sp_i)

        # 3. 피처 융합 (Concatenation)
        out = torch.cat(sp, 1)

        # 4. 최종 3x3 Conv
        out = self.conv3(out)
        out = self.bn3(out)

        # SE Block
        out = self.se(out)

        # 5. 잔차 연결 (Residual Connection)
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class Res2NetSE18_Small(nn.Module):
    def __init__(self, num_classes=10, scale=4):
        super(Res2NetSE18_Small, self).__init__()
        
        self.inplanes = 64
        self.scale = scale
        
        # 64x64 입력용 Stem (CIFAR 스타일)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # Layers (각 2개의 블록 = 18 레이어)
        self.layer1 = self._make_layer(Res2NetSEBasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(Res2NetSEBasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(Res2NetSEBasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(Res2NetSEBasicBlock, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * Res2NetSEBasicBlock.expansion, num_classes)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample, scale=self.scale))
        self.inplanes = planes * block.expansion
        
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes, scale=self.scale))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

In [11]:
class EnsembleTTA_Res2NetSE18(nn.Module):
    def __init__(self, num_models=1, num_classes=15, base_seed=SEED, device='cpu'):
        super(EnsembleTTA_Res2NetSE18, self).__init__()
        self.models = nn.ModuleList()
        self.num_classes = num_classes
        self.device = device
        
        current_rng_state = torch.get_rng_state()

        print(f"Initializing {num_models} models with different seeds...")

        for i in range(num_models):
            seed = base_seed + i
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
            
            model = Res2NetSE18_Small(num_classes=num_classes)
            
            model.to(device)
            self.models.append(model)
            print(f" - Model {i+1} initialized with seed {seed}")

        torch.set_rng_state(current_rng_state)

    def forward(self, x):
        if self.training:
            outputs = [model(x) for model in self.models]
            avg_logits = torch.stack(outputs).mean(dim=0)
            return avg_logits
        else:
            all_model_preds = []
            
            for model in self.models:
                all_model_preds.append(model(x))
            
            x_flipped = torch.flip(x, dims=[3])
            for model in self.models:
                all_model_preds.append(model(x_flipped))
            
            ensembled_logits = torch.stack(all_model_preds).mean(dim=0)
            return ensembled_logits

In [12]:
model = EnsembleTTA_Res2NetSE18().to(device)

Initializing 1 models with different seeds...
 - Model 1 initialized with seed 42


# Model parameter checking

In [13]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 12441487
Parameter usage : 12.441487%


# Model training

In [14]:
import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

In [15]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [16]:
MAX_LR_STABLE = 1e-3 # 최대 LR (0.001)
BASE_LR_MIN = 5e-5   # 최소 LR (Max_LR의 약 1/20)
SGD_MAX_LR = 0.1
SGD_WD = 5e-4 # SGD는 WD를 낮게 씁니다.

# 옵티마이저 설정 (최소 LR로 시작)
optimizer = optim.SGD(
    model.parameters(), 
    lr=BASE_LR_MIN,
    momentum=0.9, 
    weight_decay=SGD_WD
)

# 스케줄러 설정 (최대 LR로 점프)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=SGD_MAX_LR,
    epochs=epochs, 
    steps_per_epoch=len(train_loader),
    pct_start=0.3, 
    div_factor=SGD_MAX_LR / BASE_LR_MIN,
    final_div_factor=2000, 
)

criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing).to(device)
scaler = torch.amp.GradScaler("cuda")

best_val_acc = -1.0

In [17]:
print("Start Training...")

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast("cuda"):
            out = model(x)
            loss = criterion(out, y)

        scaler.scale(loss).backward()
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step() 
        
        pred = out.argmax(dim=1)        
        train_loss_sum += loss.item() * x.size(0)
        train_correct += (pred == y).sum().item()
        train_total += x.size(0)

    # epochs 통계 출력
    train_loss = train_loss_sum / max(1, train_total)
    train_acc = train_correct / max(1, train_total)
    
    # VALIDATION
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            out = model(x)
            loss = criterion(out, y)

            val_loss_sum += loss.item() * y.size(0)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / max(1, val_total)
    val_acc = val_correct / max(1, val_total)
    
    # 한 줄로 깔끔하게 출력
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    
    # Best Model 저장
    if val_acc > best_val_acc:  
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  >>> Saved Best Model (Val Acc: {val_acc:.4f})")
    
    # 마지막 모델 저장
    torch.save(model.state_dict(), "last_model.pth")

Start Training...


Epoch 01 | Train Loss: 2.4620 Acc: 0.2246 | Val Loss: 2.2120 Acc: 0.3221
  >>> Saved Best Model (Val Acc: 0.3221)


Epoch 02 | Train Loss: 2.0487 Acc: 0.3955 | Val Loss: 2.0237 Acc: 0.4084
  >>> Saved Best Model (Val Acc: 0.4084)


Epoch 03 | Train Loss: 1.8638 Acc: 0.4711 | Val Loss: 2.1347 Acc: 0.4037


Epoch 04 | Train Loss: 1.7271 Acc: 0.5249 | Val Loss: 1.7795 Acc: 0.5028
  >>> Saved Best Model (Val Acc: 0.5028)


...

Epoch 26 | Train Loss: 0.9218 Acc: 0.8558 | Val Loss: 1.0685 Acc: 0.7889
  >>> Saved Best Model (Val Acc: 0.7889)


Epoch 27 | Train Loss: 0.8415 Acc: 0.8886 | Val Loss: 1.0055 Acc: 0.8214
  >>> Saved Best Model (Val Acc: 0.8214)


Epoch 28 | Train Loss: 0.7622 Acc: 0.9249 | Val Loss: 0.9610 Acc: 0.8362
  >>> Saved Best Model (Val Acc: 0.8362)


Epoch 29 | Train Loss: 0.6993 Acc: 0.9531 | Val Loss: 0.9436 Acc: 0.8500
  >>> Saved Best Model (Val Acc: 0.8500)


Epoch 30 | Train Loss: 0.6724 Acc: 0.9659 | Val Loss: 0.9339 Acc: 0.8502
  >>> Saved Best Model (Val Acc: 0.8502)


In [18]:
model.load_state_dict(torch.load(save_path, map_location=device))

C:\Users\user\AppData\Local\Temp\ipykernel_17308\1058478286.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path, map_location=devi

<All keys matched successfully>

# Submit
Do not edit the submission code below.

In [19]:
submit = pd.read_csv('./input/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 12441487
Parameter usage : 12.441487%


100%|██████████| 30/30 [00:08<00:00,  3.42it/s]
